In [51]:
import requests

payload = {
    "question": "what is economics?",
    "conv_id": "test_conv_123"
}

response = requests.post("http://127.0.0.1:8003/process", json=payload)

In [52]:
response.json()

{'plan_summary': "The user is asking for a definition of 'economics'. I will use a SEARCH_RAG node to find information about the definition of economics and then a SYNTHESIZE node to formulate the answer.",
 'nodes': [{'id': 'node_1',
   'type': 'SEARCH_RAG',
   'search_query': 'definition of economics',
   'prompt_template': None,
   'question_to_ask': None},
  {'id': 'node_2',
   'type': 'SYNTHESIZE',
   'search_query': None,
   'prompt_template': 'Based on the information from {node_1}, what is economics? Provide a concise definition.',
   'question_to_ask': None}],
 'edges': [{'source': 'node_1', 'target': 'node_2'}]}

In [61]:
import requests
import json
import time

URL = "http://127.0.0.1:8003/process"

test_questions = [
    # 1. Easy / Informational
    {"category": "Easy Direct", "question": "What is macroeconomics?"},
    
    # 2. Ambiguous / Clarification Needed
    {"category": "Ambiguous Input", "question": "How much does it cost?"},
    
    # 3. Multi-hop / Complex Analytical
    {"category": "Complex Analytical", "question": "Compare inflation dynamics in 2022 versus 1970 and synthesize the key monetary policy differences."},
    
    # 4. Sequential Dependency
    {"category": "Multi-Step Process", "question": "Decompose the causes of supply chain shocks, search RAG for historical data, and ask me if I want a summarized chart."}
]

print("--- STARTING GRAPH PLANNER BENCHMARK WITH LATENCY TIMING ---\n")

total_benchmark_start = time.perf_counter()

for item in test_questions:
    payload = {"question": item["question"], "conv_id": "test_suite_01"}
    
    start_time = time.perf_counter()
    try:
        response = requests.post(URL, json=payload, timeout=60)
        elapsed_time = time.perf_counter() - start_time
        
        if response.status_code == 200:
            print(f"=== Category: {item['category']} ===")
            print(f"Question: '{item['question']}'")
            print(f"⏱️ Response Time: {elapsed_time:.3f} seconds")
            print("Generated Graph Blueprint:")
            print(json.dumps(response.json(), indent=2))
            print("-" * 50 + "\n")
        else:
            print(f"Failed [{response.status_code}] ({elapsed_time:.3f}s): {response.text}")
    except Exception as e:
        elapsed_time = time.perf_counter() - start_time
        print(f"Request failed ({elapsed_time:.3f}s): {e}")

total_benchmark_time = time.perf_counter() - total_benchmark_start
print(f"=== BENCHMARK COMPLETE (Total Time: {total_benchmark_time:.3f}s) ===")

--- STARTING GRAPH PLANNER BENCHMARK WITH LATENCY TIMING ---

=== Category: Easy Direct ===
Question: 'What is macroeconomics?'
⏱️ Response Time: 3.306 seconds
Generated Graph Blueprint:
{
  "nodes": [
    {
      "id": "node_1",
      "type": "SEARCH_RAG",
      "search_query": "define macroeconomics",
      "prompt_template": null,
      "question_to_ask": null
    },
    {
      "id": "node_2",
      "type": "SYNTHESIZE",
      "search_query": null,
      "prompt_template": "Summarize the definition of macroeconomics from {node_1}.",
      "question_to_ask": null
    }
  ],
  "edges": [
    {
      "source": "node_1",
      "target": "node_2"
    }
  ]
}
--------------------------------------------------

=== Category: Ambiguous Input ===
Question: 'How much does it cost?'
⏱️ Response Time: 0.909 seconds
Generated Graph Blueprint:
{
  "nodes": [
    {
      "id": "node_1",
      "type": "CLARIFY",
      "search_query": null,
      "prompt_template": null,
      "question_to_ask": "W